In [1]:
import argparse
import itertools
import os
import pathlib
import sys
from functools import reduce

import duckdb
import pandas as pd
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm
profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir  # default to root_dir instead of NAS

In [2]:
patient_id_file = pathlib.Path(f"{profile_base_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patient_id_file, header=None, names=["patient_id"]
).patient_id.tolist()

In [3]:
out_dict = {
    "file_path": [],
    "patient_id": [],
    "well_fov": [],
    "feature_type": [],
    "compartment": [],
    # "df_shape": [],
}

# get all well_fovs for a patient
for patient in tqdm.tqdm(patients, desc="Processing patients", leave=True):
    patient_dir = profile_base_dir / "data" / patient / "extracted_features"
    well_fovs = patient_dir.glob("*")  # get all well_fovs for a patient
    # print(f"Found well_fovs: {well_fovs}")
    for well_fov in tqdm.tqdm(well_fovs, desc="Processing well_fovs", leave=False):
        if "stats" in well_fov.stem:
            continue
        features = pathlib.Path(well_fov).glob("*.parquet")
        for feature in features:
            feature_type = feature.stem.split("_")[2]
            compartment = feature.stem.split("_")[0]
            out_dict["file_path"].append(feature)
            out_dict["patient_id"].append(patient)
            out_dict["well_fov"].append(feature.parent.stem)
            out_dict["feature_type"].append(feature_type)
            out_dict["compartment"].append(compartment)
            # out_dict["df_shape"].append(pd.read_parquet(feature).shape)
df = pd.DataFrame(out_dict)
df = df.loc[df["patient_id"] == "NF0014_T1"]
# df = df.loc[(df["feature_type"] == "AreaSizeShape") & (df["compartment"] != "Organoid")]
df = df.loc[(df["compartment"] != "Organoid")]

df.head()

Processing patients:   0%|          | 0/13 [00:00<?, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

...

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

,file_path,patient_id,well_fov,feature_type,compartment
0,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,SAMMed3D,Nuclei
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
2,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Intensity,Cytoplasm
3,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Intensity,Nuclei
5,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm


In [4]:
from tqdm import tqdm

tqdm.pandas()


def safe_read_shape(x):
    try:
        df = pd.read_parquet(x)
        return df.shape, df.isna().sum().sum()
    except Exception as e:
        print(f"Error reading {x}: {e}")
        return None, None


# if not pathlib.Path("../logs/feature_file_info.parquet").exists():
df[["df_shape", "missing_values"]] = df["file_path"].progress_apply(
    lambda x: pd.Series(safe_read_shape(x))
)
df["file_path"] = df["file_path"].astype(str)
df.to_parquet("../logs/feature_file_info.parquet", index=False)
# else:
#     df = pd.read_parquet("../logs/feature_file_info.parquet")

100%|██████████| 8061/8061 [01:12<00:00, 111.87it/s]


In [5]:
df.sort_values(["patient_id", "well_fov"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [6]:
# merge the cells, cytoplasm, and whole cell features for a given well_fov and patient_id
# check for missing values and shape of the dataframes
out_dict = {
    "patient_id": [],
    "well_fov": [],
    "path": [],
    "type": [],
}
for row in tqdm(
    df.itertuples(), total=df.shape[0], desc="Merging features", leave=True
):
    out_dict["patient_id"].append(row.patient_id)
    out_dict["well_fov"].append(row.well_fov)
    out_dict["path"].append(row.file_path)
    out_dict["type"].append(f"{row.compartment}")
out_df = pd.DataFrame(out_dict)
# pivot such that each type has its own column
out_df = out_df.pivot(
    index=["patient_id", "well_fov"], columns="type", values="path"
).reset_index()

Merging features: 100%|██████████| 8061/8061 [00:00<00:00, 839673.29it/s]


ValueError: Index contains duplicate entries, cannot reshape

In [ ]:
labels_dict = {
    "Cell_labels": [],
    "Cytoplasm_labels": [],
    "Nuclei_labels": [],
    "patient_id": [],
    "well_fov": [],
}

# merge the dataframes and check for missing values and shape
for row in tqdm(
    out_df.itertuples(),
    total=out_df.shape[0],
    desc="Checking merged features",
    leave=True,
):
    try:
        cell_df = pd.read_parquet(row.Cell)
        cytoplasm_df = pd.read_parquet(row.Cytoplasm)
        nuclei_df = pd.read_parquet(row.Nuclei)
        labels_dict["Cell_labels"].append(cell_df["object_id"].tolist())
        labels_dict["Cytoplasm_labels"].append(cytoplasm_df["object_id"].tolist())
        labels_dict["Nuclei_labels"].append(nuclei_df["object_id"].tolist())
        labels_dict["patient_id"].append(row.patient_id)
        labels_dict["well_fov"].append(row.well_fov)
    except Exception as e:
        print(f"Error reading files for {row.patient_id} {row.well_fov}: {e}")
labels_df = pd.DataFrame(labels_dict)

Checking merged features:  69%|██████▉   | 71/103 [00:00<00:00, 233.41it/s]

Error reading files for NF0014_T1 E10-1: cannot construct a FileSource from nan


Checking merged features: 100%|██████████| 103/103 [00:00<00:00, 231.96it/s]


In [ ]:
labels_df["labels_match"] = labels_df.apply(
    lambda row: row["Cell_labels"] == row["Nuclei_labels"],
    axis=1,
)
labels_df["same_number_of_labels"] = labels_df.apply(
    lambda row: len(row["Cell_labels"]) == len(row["Nuclei_labels"]),
    axis=1,
)
labels_df["unique_labels_across_compartments"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cytoplasm_labels"]), axis=1
)
labels_df.loc[labels_df["labels_match"] == False]

,Cell_labels,Cytoplasm_labels,Nuclei_labels,patient_id,well_fov,labels_match,same_number_of_labels,unique_labels_across_compartments


In [ ]:
labels_df.loc[labels_df["same_number_of_labels"] == False].value_counts("patient_id")

Series([], Name: count, dtype: int64)

In [ ]:
# # show the well fov and the patient id
# # print the unique well fovs that have mismatched labels
# for row in labels_df.loc[labels_df["labels_match"] == False][
#     ["patient_id", "well_fov"]
# ].itertuples():
#     print(f"cd ../../{row.patient_id}/extracted_features/ ; rm -r {row.well_fov}")

In [ ]:
tmp_df = pd.merge(
    left=pd.merge(
        left=cell_df,
        right=cytoplasm_df,
        on=["object_id", "image_set"],
    ),
    right=nuclei_df,
    on=["object_id", "image_set"],
)
tmp_df.head()

,object_id,image_set,Cell_NoChannel_AreaSizeShape_Volume,Cell_NoChannel_AreaSizeShape_CenterX,Cell_NoChannel_AreaSizeShape_CenterY,Cell_NoChannel_AreaSizeShape_CenterZ,Cell_NoChannel_AreaSizeShape_BboxVolume,Cell_NoChannel_AreaSizeShape_MinX,Cell_NoChannel_AreaSizeShape_MaxX,Cell_NoChannel_AreaSizeShape_MinY,...,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,Nuclei_NoChannel_AreaSizeShape_MaxY,Nuclei_NoChannel_AreaSizeShape_MinZ,Nuclei_NoChannel_AreaSizeShape_MaxZ,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_SurfaceArea
0,1,G9-2,160340.0,903.442647,1032.891106,5.328022,651015.0,815,1000,956,...,840,954,989,1102,0,6,0.682205,1,46.524706,188.114624
1,2,G9-2,150694.0,929.346132,1095.430329,11.859391,1051200.0,848,994,1009,...,877,959,1018,1180,1,9,0.219587,2,35.454873,385.502356
2,3,G9-2,255572.0,643.951849,899.223788,8.399457,764244.0,525,732,820,...,591,697,839,935,2,17,0.393003,1,48.568563,426.411935
3,4,G9-2,345522.0,612.952877,891.132886,32.857578,1158300.0,519,717,822,...,591,680,839,916,17,19,0.766672,1,27.175267,24.427922
4,5,G9-2,217512.0,597.274210,1165.513471,10.117198,613080.0,535,666,1096,...,552,643,1111,1202,6,21,0.557437,1,50.947592,432.996470
